# OpenML · Phase 2 — labeling + data versioning
1. Generate a small image dataset into the **shared workspace** (visible to both Jupyter and Label Studio).
2. Label it in **Label Studio** (:8081) using *Local Storage* — no S3 presigned-URL headaches.
3. Version the dataset with **DVC** and push it to **MinIO**.

Turn the **Label** stack on in the console (:8080) first.

In [ ]:
import os, random, numpy as np
from PIL import Image, ImageDraw
import openml_telemetry as tel

root = '/home/jovyan/work/workspace/labeling'
imgdir = os.path.join(root, 'images'); os.makedirs(imgdir, exist_ok=True)
shapes = ['circle', 'square', 'triangle']; random.seed(0); np.random.seed(0)
counts = {s: 0 for s in shapes}
for i in range(12):
    shape = shapes[i % 3]; counts[shape] += 1
    img = Image.new('RGB', (128, 128), (28, 30, 40)); d = ImageDraw.Draw(img)
    color = tuple(int(x) for x in np.random.randint(90, 255, 3))
    if shape == 'circle':   d.ellipse([24, 24, 104, 104], fill=color)
    elif shape == 'square': d.rectangle([28, 28, 100, 100], fill=color)
    else:                   d.polygon([(64, 22), (104, 104), (24, 104)], fill=color)
    img.save(f'{imgdir}/{shape}_{i:02d}.png')
print('wrote', len(os.listdir(imgdir)), 'images ->', imgdir)

# publish data-prep stats to Grafana 'Data Prep'
tel.push_dataset_stats('shapes', rows=12, classes=counts, splits={'unlabeled': 12})
print('class counts:', counts)

## Label in Label Studio (:8081)
Login: `admin@openml.local` / `openml-admin` (from `.env`).

1. **Create Project** → name it *shapes*.
2. **Settings → Labeling Interface → Code**, paste:
   ```xml
   <View>
     <Image name="image" value="$image"/>
     <Choices name="label" toName="image">
       <Choice value="circle"/><Choice value="square"/><Choice value="triangle"/>
     </Choices>
   </View>
   ```
3. **Settings → Cloud Storage → Add Source Storage → Local files**
   * Absolute local path: `/label-studio/files/labeling/images`
   * *Treat every bucket object as a source file* = on → **Sync**.
4. Label the images, then **Export → JSON** into `workspace/labeling/exports/`.

> Local files work because Label Studio serves them itself — the images live in the
> shared `workspace` volume you just wrote to. (S3/MinIO source storage also works,
> but needs the browser to resolve `minio`; see `docs/`.)

In [ ]:
# --- version the dataset with DVC and push to MinIO ---
%cd /home/jovyan/work/workspace
!dvc init --no-scm -f -q
!dvc remote add -d -f minio s3://datasets/dvc
!dvc remote modify minio endpointurl http://minio:9000
!dvc add labeling
!dvc push
print('DVC pushed labeling/ -> MinIO s3://datasets/dvc')

In [ ]:
# --- verify the versioned data really landed in MinIO ---
import boto3, os
s3 = boto3.client('s3', endpoint_url=os.environ['OPENML_S3_ENDPOINT'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'])
objs = s3.list_objects_v2(Bucket='datasets', Prefix='dvc/').get('Contents', [])
print('DVC objects in MinIO s3://datasets/dvc:', len(objs))
print('pointer file labeling.dvc:\n', open('labeling.dvc').read())

### Recap
* **Label Studio** :8081 — annotate; data comes from the shared `workspace`.
* **MinIO** :9001 — `datasets/dvc/…` now holds the content-addressed dataset.
* **Grafana → Data Prep** — class balance / row counts you pushed.
* `labeling.dvc` is the tiny pointer you'd commit to git; `dvc pull` restores the data anywhere.